# 1. Azure Identity Concepts & Tokens

Before we call APIs, let's get the vocabulary straight. Azure's identity world has *way* too many terms and most tutorials assume you already know them. We'll learn them by poking at a live mock Entra ID server.

## The four things you need to know

| Term | What it really is | Real-world analogy |
|------|-------------------|--------------------|
| **Tenant** | An Entra directory — contains users, groups, app registrations. Has a GUID. | A company building |
| **App registration** | The *definition* of an application (client ID, secrets, what scopes it exposes). | A job posting describing a role |
| **Service principal** | An *instance* of that app inside a tenant. What actually holds permissions. | A specific person hired for that role |
| **Access token** | A signed JWT proving the caller's identity + permissions. | A badge you swipe at the door |

> In a single tenant, every app registration has one service principal in the same tenant. For multi-tenant apps, admins in other tenants "consent" and a service principal for your app is created in *their* tenant. They control what permissions you get there.

## Setup

```bash
cd 08-enterprise/azure-authentication
docker compose up -d
```

Then pick the lab's `.venv` interpreter as the kernel (top-right in VS Code it shows as
`Python 3 (.venv)`). If it's missing, reload the window (`Cmd+Shift+P` → **Reload Window**).

In [ ]:
import httpx, json, base64

AUTHORITY = 'http://localhost:9100/contoso'
TOKEN_URL = f'{AUTHORITY}/oauth2/v2.0/token'

# The OIDC discovery document - every identity provider publishes one.
# SDKs call this to learn the token endpoint, JWKS URI, supported grants.
d = httpx.get(f'{AUTHORITY}/v2.0/.well-known/openid-configuration').json()
print(json.dumps(d, indent=2))

**Why this matters:** when you configure a Python SDK or FastAPI JWT validator, you point it at the issuer URL and it fetches this document automatically. In real Entra ID the URL is:

```
https://login.microsoftonline.com/<tenant-id>/v2.0/.well-known/openid-configuration
```

## App registrations — what's inside one?

Look at `fake-entra/apps.json` — each entry mirrors what you'd see on the **App registrations → overview** blade in the Azure portal:

- `client_id` — the public identifier. Like a username for the app.
- `client_secret` — a password. Alternative: upload a certificate.
- `identifier_uri` (a.k.a. *Application ID URI*) — how *other* apps address this one when requesting a token. Set on the **Expose an API** blade.
- `exposed_scopes` — delegated permissions (user acts through the app): `Files.Read`, `Mail.Send`.
- `app_roles` — application permissions (app acts on its own): `Files.Read.All`.
- `granted_app_roles` — what app-roles *this* app has been granted on *other* apps. In Azure this is configured under **API permissions** and requires admin consent.

In [ ]:
print(json.dumps(json.load(open('../fake-entra/apps.json')), indent=2))

## Anatomy of an access token (JWT)

An access token is three base64url-encoded pieces separated by dots: `header.payload.signature`.

- **header**: algorithm (RS256), key ID (`kid`) — tells validators which public key to use.
- **payload**: the claims — issuer (`iss`), audience (`aud`), subject (`sub`), expiry (`exp`), plus custom claims like `scp` (scopes) or `roles` (app roles).
- **signature**: RSA signature over header+payload. Validators verify it with the public key from JWKS.

> ## ⛔ DECODING IS NOT VALIDATING ⛔
>
> The next cell splits the token on `.` and base64-decodes it. **That is not a security
> check.** Base64 is an encoding, not encryption and not a signature — anyone can type
> `{"roles":["Global.Admin"],"upn":"ceo@contoso.com"}`, base64 it, staple on any old
> third segment and it will "decode" perfectly.
>
> A token is only trustworthy after a validator has, *in this order*:
> 1. verified the **RS256 signature** against the tenant's **JWKS** public key named by `kid`;
> 2. checked `iss` is your tenant, `aud` is *this* API, and `exp` / `nbf` put "now" inside
>    the validity window (with a minute or so of clock skew).
>
> `decode_jwt` below exists **only** so we can look at claim shapes on screen. The code
> that actually gates access lives in [`common/auth.py`](../common/auth.py), and it does all
> of the above. Never branch on a claim you have not verified.

Let's get a token and look at it.

In [ ]:
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'client_credentials',
    'client_id': 'daemon-client-id',
    'client_secret': 'daemon-secret-value',
    'scope': 'api://api-b/.default',
})
token = r.json()['access_token']
print(token[:60] + '...')

In [ ]:
# ⛔ INSPECTION ONLY - this function performs NO signature check and NO claim
# validation. Its output tells you what a token *claims*, never whether that
# claim is true. See common/auth.py for the real thing.
def decode_jwt(tok):
    header_b64, payload_b64, _sig = tok.split('.')   # note: _sig is thrown away
    pad = lambda s: s + '=' * (-len(s) % 4)
    return {
        'header': json.loads(base64.urlsafe_b64decode(pad(header_b64))),
        'payload': json.loads(base64.urlsafe_b64decode(pad(payload_b64))),
    }

print(json.dumps(decode_jwt(token), indent=2))

### Key claims to know

| Claim | Meaning | Why validators check it |
|-------|---------|-------------------------|
| `iss` | Issuer URL | Is this token from *our* Entra tenant? |
| `aud` | Audience (the API this token is *for*) | Prevents tokens issued for one API being replayed at another |
| `exp` | Expiry (unix seconds) | Short-lived (usually 1 hour) limits damage if stolen |
| `nbf` | Not-before (unix seconds) | Rejects a token presented before its validity window opens |
| `iat` | Issued-at (unix seconds) | Age of the token; useful for auditing |
| `sub` / `oid` | Who the token is about | `oid` is the stable object ID in Entra |
| `azp` | Authorized party — the *client* that requested the token | Useful for logging |
| `scp` | Space-separated **delegated** scopes | User-signed-in flows |
| `roles` | Array of **application** roles | App-only flows |
| `tid` | Tenant ID | Multi-tenant apps use this |

> **A claim that is absent is not a claim that is false.** Several JWT libraries —
> `python-jose` among them — *skip* validating `aud` or `exp` when the claim simply isn't
> in the token, so a signed token with no `aud` and no `exp` passes a naive
> `jwt.decode(token, key, audience=..., issuer=...)` as "valid for any audience, forever".
> [`common/auth.py`](../common/auth.py) closes that hole by passing
> `options={"require_aud": True, "require_exp": True, ...}`. Check your own library for
> the same behaviour before you trust it.

## ID token vs access token — the distinction people get wrong

They are both JWTs signed by the same tenant, and they look almost identical when you
decode them. They are for completely different jobs:

| | **ID token** | **Access token** |
|-|--------------|------------------|
| Answers | "*Who signed in?*" | "*What is the bearer allowed to do here?*" |
| Audience (`aud`) | Your **client app's** client ID | The **resource API's** identifier URI (e.g. `api://api-b`) |
| Intended consumer | The **client** that requested sign-in | The **API** named in `aud` |
| Typical claims | `name`, `preferred_username`, `email`, `nonce`, `amr` | `scp` or `roles`, `azp`, `oid` |
| Spec | OpenID Connect | OAuth 2.0 |
| Put it in `Authorization: Bearer`? | ❌ never | ✅ that's what it's for |

Two rules that follow, and both are load-bearing:

1. **An API must never accept an ID token as its bearer credential.** An ID token's `aud`
   is the *client*, so accepting one means accepting a token minted for somebody else's
   audience — and ID tokens carry no `scp`/`roles`, so there is nothing to authorize
   against. A correct validator rejects it automatically *because* it checks `aud`. This
   is the number-one reason `aud` validation is non-negotiable.
2. **A client must never inspect an access token to decide what to show the user.** The
   access token is opaque to the client by contract; Entra may change its format. Read
   the ID token (or call `/me`) for user info.

Everything in this lab issues **access tokens** — none of the grants we use (client
credentials, OBO) produce an ID token, because none of them are sign-in flows for a
client app.

## JWKS — how validators trust the signature

Your API never has a shared secret with Entra. Instead Entra publishes its **public keys** at a well-known JWKS URL. Your API fetches + caches them and verifies the signature.

In [ ]:
jwks = httpx.get(d['jwks_uri']).json()
print(json.dumps(jwks, indent=2))
print('\nToken header kid:', decode_jwt(token)['header']['kid'])
print('JWKS key kid    :', jwks['keys'][0]['kid'])

## Bad → best: why bother with all this?

A common first reaction is *"couldn't I just use a shared API key?"* Let's walk the ladder.

### ❌ Level 0 — Shared API key in a header

```http
GET /files
X-Api-Key: super-secret-string
```

Problems:
- Any service that has ever seen the key can impersonate every caller. No way to tell *who* is calling.
- Rotating the key means redeploying every caller at once.
- Stolen from logs / git / a curl copy-paste → full access, no expiry.
- No fine-grained permissions. It's “all or nothing”.

### ⚠️ Level 1 — JWT we sign ourselves with a shared HMAC secret

Better (expiry, claims, per-caller `sub`), but:
- Every verifier needs the **same** secret. A leak at any verifier = forged tokens anywhere.
- You own the key rotation story, the revocation story, the clock-skew story, the algorithm-confusion story (`alg: none` attacks, HS256/RS256 confusion…).
- Writing an identity provider is how people end up with CVEs.

### ✅ Level 2 — Entra-issued RS256 JWT + JWKS validation (what this lab teaches)

- **Asymmetric keys.** Entra holds the private key; your APIs only need the *public* JWKS. Leaking an API container does not let the attacker forge tokens.
- **Short-lived** (1h default). Stolen tokens expire quickly.
- **Audience-scoped** (`aud`). A token for API-B cannot be replayed against API-A.
- **Per-caller identity.** `sub`/`oid` tells you exactly which service principal or user called.
- **Fine-grained permissions.** `scp` (delegated) and `roles` (app) let you express “read own files” vs “read all files” without custom logic.
- **Automatic key rotation** via `kid` → JWKS. You do nothing; validators re-fetch.

Every technique in the next notebooks (client credentials, managed identity, OBO) is a *variation* of this level-2 model.


In [ ]:
# Proof-by-code: tokens are *signed*, not just base64. We forge a token that decodes
# perfectly and claims to be a tenant admin -- and watch api-b throw it out.
import httpx

# Grab a valid token.
t = httpx.post(TOKEN_URL, data={
    'grant_type': 'client_credentials',
    'client_id': 'daemon-client-id',
    'client_secret': 'daemon-secret-value',
    'scope': 'api://api-b/.default',
}).json()['access_token']

# Valid call against api-b succeeds.
ok = httpx.get('http://localhost:8002/files', headers={'Authorization': f'Bearer {t}'})
print('original token ->', ok.status_code)
assert ok.status_code == 200, f'expected a valid token to be accepted, got {ok.status_code} {ok.text}'

# Forge: rewrite the payload to grant ourselves an app role we were never assigned,
# then staple the ORIGINAL signature back on. Nothing stops us writing the bytes.
h, pl, sig = t.split('.')
forged_payload = {**decode_jwt(t)['payload'], 'roles': ['Global.Admin'], 'sub': 'ceo@contoso.com'}
b64 = base64.urlsafe_b64encode(json.dumps(forged_payload, separators=(',', ':')).encode()).rstrip(b'=').decode()
bad = f'{h}.{b64}.{sig}'
assert bad != t, 'forgery was a no-op - the demo would prove nothing'

# It decodes flawlessly. Anything that trusts a *decoded* claim just handed over the tenant.
print('forged token decodes to roles =', decode_jwt(bad)['payload']['roles'])
assert decode_jwt(bad)['payload']['roles'] == ['Global.Admin'], 'the forged claim should decode fine'

# But the signature no longer covers those bytes, so the *validator* rejects it.
tampered = httpx.get('http://localhost:8002/files', headers={'Authorization': f'Bearer {bad}'})
print('forged token ->', tampered.status_code, tampered.json())
assert tampered.status_code == 401, (
    f'a re-signed-payload forgery MUST be rejected with 401, got {tampered.status_code}. '
    'If this ever passes, the API is not verifying signatures and every claim is attacker-controlled.'
)

## Which OAuth2 flow for which caller?

The ladder above is about *token format*. This table is about *how you get one*. Picking the
wrong grant is the other half of getting Entra right.

| Caller | Flow | Notes |
|--------|------|-------|
| **Public client** — SPA, mobile app, desktop app, CLI | **Authorization code + PKCE** | The answer. A public client cannot keep a secret (its binary/JS ships to the user), so it has none; PKCE (RFC 7636) binds the redirect back to the process that started it, defeating code interception. Entra **requires** PKCE for SPAs. |
| **Confidential client** — web app with a server-side backend | Authorization code (+ PKCE, still recommended) with a client secret **or certificate** | The secret lives on the server, never in the browser. |
| **Daemon / worker / cron** — no user present | **Client credentials** | Notebook 2. |
| **Code running on Azure compute** | **Managed identity** | Notebook 3. Client credentials with the credential handled by the platform. |
| **Middle-tier API calling another API as the user** | **On-Behalf-Of** | Notebook 4. |
| **Input-constrained device** — TV, IoT, headless CLI | Device code | User authenticates on a second device. |

### Two flows you should recognise in order to refuse them

- **Implicit flow** (`response_type=token` / `id_token token`) — **deprecated.** It returned
  access tokens in the URL fragment, so they landed in browser history, `Referer` headers and
  server logs, and it has no refresh-token story. The OAuth 2.0 Security BCP (RFC 9700) says
  do not use it; OAuth 2.1 removes it. Authorization code + PKCE replaced it, and it works in
  SPAs today thanks to CORS on the token endpoint. If you meet implicit flow in an existing
  app, that is a migration ticket, not a pattern to copy.
- **Resource owner password credentials (ROPC)** — the `grant_type=password` our mock
  implements, and which notebook 4 uses purely to conjure a user token without a browser.
  It requires your app to *handle the user's actual password*, so it cannot do MFA,
  federation, Conditional Access or passwordless sign-in. Microsoft advises against it, it is
  blocked for personal accounts and for any account with MFA, and OAuth 2.1 removes it too.
  We use it here because a notebook cannot do a browser redirect — **not** because it is OK.

The `kid` in the token header matches the `kid` in JWKS — that's how the validator picks the right public key when Entra rotates keys.

## What's next

- **Notebook 2** — use this token to call `api-b` (client credentials / S2S).
- **Notebook 3** — *managed identities*: how Azure gives your container a token with no secrets in code.
- **Notebook 4** — *On-Behalf-Of*: user → api-a → api-b with user identity preserved end-to-end.
- **Notebook 5** — local dev with `DefaultAzureCredential`.